In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np   # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
'''
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
'''
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

"\nimport os\nfor dirname, _, filenames in os.walk('/kaggle/input'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n"

# Dataset

In [5]:
movies = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')

In [6]:
movies.head(1).shape

(1, 20)

In [7]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [8]:
movies = movies.merge(credits, on='title')  # Both datasets merged to form 4809 rows x 23 cols
print(movies)

         budget                                             genres  \
0     237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1     300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2     245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3     250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4     260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
...         ...                                                ...   
4804     220000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4805       9000  [{"id": 35, "name": "Comedy"}, {"id": 10749, "...   
4806          0  [{"id": 35, "name": "Comedy"}, {"id": 18, "nam...   
4807          0                                                 []   
4808          0                [{"id": 99, "name": "Documentary"}]   

                                               homepage      id  \
0                           http://www.avatarmovie.com/   19995   
1          http://disney.

In [9]:
movies.info()
# genres, keywords, overview, title, movie_id, cast, crew  (optional - release date, revenue, runtime, status)
# original_lang not imp - movies['original_language'].value_counts() shows 4510/5000 are Eng movies - so other lang are insignificant
# popularity not imp - just a number, its based on what?

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [10]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

Now we will merge 'overview', 'genres', 'keywords', 'cast', 'crew' as tags. But we will have to preprocess these columns, as they are in json format. 1st lets check if there are any missing data.

In [11]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [12]:
movies.dropna(inplace=True)  # Since 3/5000 is missing, its insignificant, so we delete them

In [13]:
movies.duplicated().sum()  # No duplicates

np.int64(0)

In [14]:
movies.iloc[0].genres  # Finding loc by index

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [15]:
# We have to convert [{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}] json format 
# to ["Action", "Adventure", "Fantasy", "Science Fiction"]
import ast

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):  # Safely converts a string into a Python object
        L.append(i['name'])
    return L

In [16]:
movies['keywords'] = movies['keywords'].apply(convert)

In [17]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):  # Safely converts a string into a Python object
        if counter != 3:             # Bcoz we want the top 3 names of actors.
            L.append(i['name'])
            counter+=1
        else:
            break
    return L

In [18]:
movies['genres'] = movies['genres'].apply(convert3)

In [19]:
movies['cast'] = movies['cast'].apply(convert3)

In [20]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):  # Safely converts a string into a Python object
        if i['job'] == 'Director':             # Bcoz we want the top 3 names of actors.
            L.append(i['name'])
            break
    return L

In [21]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [22]:
movies['overview'] = movies['overview'].apply(lambda x:x.split()) # To split the string into a list of words

In [23]:
# To remove spaces between names, so that it acts like a single tag(entity)
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [24]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton]


In [25]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [26]:
new_df = movies[['movie_id', 'title', 'tags' ]]

In [27]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x)) # convert list into a string

/tmp/ipykernel_55/4014975321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x)) # convert list into a string


In [28]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower()) # To make everything Lower Case

/tmp/ipykernel_55/70009826.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower()) # To make everything Lower Case


In [29]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


Now to suggest top 5 movies, we will have to find similarity between the tags. Instead we will use the concept of Text Vectorisation to convert the text data into numerical feature vectors using Bag of Words (BoW).

In [30]:
from sklearn.feature_extraction.text import CountVectorizer
# Builds a vocabulary of 5000 words, Counts how often each word appears & Converts text into vectors. 
cv = CountVectorizer(max_features = 5000, stop_words = 'english') # stop_words Removes common useless words like: the, is, and, of, to, in

In [31]:
vectors = cv.fit_transform(new_df['tags']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [32]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [33]:
stem('in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron')

NameError: name 'stem' is not defined

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

So now `similarity` is a 4806x4806 Matrix. In the 1st movie, similarity with itself is represented as 1. So to recommend top 5 similar movies, we will need the index of movie, then we will sort the values in that row in an ascending order and select the 1st 5 values in it.

In [36]:
similarity[0]

array([1.        , 0.08980265, 0.05986843, ..., 0.02512595, 0.02635231,
       0.        ])

But while sorting, we don't want to lose original index in sorted(similarity[0], reverse=True). So we use enumerate() to convert list into list of tuples and sort it in descending order. But now the list is sorted based on the key(index), So to sort based on the values, we use lambda x : x[1]. Then we select 1st 5 values. 

Note: Movie index starts from 0. [1:6] means starting from 1 and stopping before 6.

In [37]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key = lambda x:x[1])[1:6] 

    for i in movies_list:
        
        print(new_df.iloc[i[0]].title)  # prints the original index of movies

In [38]:
recommend('Avatar')

3728
1214
507
1440
1916
